# 06 – Pricing with QuantLib: verification and constant volatility against the SVI surface

This notebook prices the same option with QuantLib, an open-source library widely used in the
industry, for two reasons:

1. **Verification.** With the same inputs, QuantLib and our own pricer (notebook 05) must give the
   same price.
2. **Value of the surface.** With the same engine, we compare a constant volatility (Black-Scholes
   with one volatility for all strikes) with the SVI surface built in notebook 04.

The option is a 1-year European put with strike at 90% of the spot, a typical leg of an equity hedge.

In [1]:
import os
import numpy as np
import pandas as pd
from QuantLib import *
from functions import black_price, svi      # imported after QuantLib, so that no name is overwritten

SNAPSHOT = "SPX_2026-10-26_1030"
DATA_DIR = "data"

## 1. Market data and a constant volatility (case A)

All inputs come from the previous notebooks (snapshot of 26 October 2026):

- spot $S$ = 7673.61;
- rate $r$ = 4.371%, the 1-year Treasury rate converted to continuous compounding (notebook 05);
- dividend yield $q = r - \ln(F/S)$ = 0.377%, chosen so that QuantLib uses exactly our 1-year forward
  $F$ = 7986.32. It contains dividends and any other carry implied by the option prices;
- volatility 16.70%, the 1-year at-the-money volatility of the SVI surface.

The evaluation date is fixed at the snapshot date, so that the results do not change with the day
on which the notebook is run.

In [2]:
today = Date(26, 10, 2026)
Settings.instance().evaluationDate = today

S = 7673.61
K = 0.9 * S                        # put with strike at 90% of the spot

u = SimpleQuote(S)                 # spot
r = SimpleQuote(0.04371)           # 1-year Treasury rate (FRED), continuous compounding
q = SimpleQuote(0.00377)           # dividend yield implied by our forward: r - ln(F/S)
sigma = SimpleQuote(0.1670)        # constant vol: 1-year ATM vol of the SVI surface

dc = Actual365Fixed()
riskFreeTS = YieldTermStructureHandle(FlatForward(today, QuoteHandle(r), dc))
dividendTS = YieldTermStructureHandle(FlatForward(today, QuoteHandle(q), dc))
volTS = BlackVolTermStructureHandle(BlackConstantVol(today, NullCalendar(), QuoteHandle(sigma), dc))

process = BlackScholesMertonProcess(QuoteHandle(u), dividendTS, riskFreeTS, volTS)

option = EuropeanOption(PlainVanillaPayoff(Option.Put, K),
                        EuropeanExercise(Date(26, 10, 2027)))

engine = AnalyticEuropeanEngine(process)
option.setPricingEngine(engine)
print("QuantLib, constant vol:", option.NPV())

QuantLib, constant vol: 125.65153353533704


**Check.** Our own Black formula (`functions.py`) with the same inputs must give the same price.

In [3]:
F = S * np.exp((0.04371 - 0.00377) * 1.0)    # forward in 1 year
print("Our Black formula:     ", black_price(F, K, 1.0, 0.04371, 0.1670, "P"))

Our Black formula:      125.65153353533769


The two prices agree to 13 decimal places, so the QuantLib set-up (dates, day count, curves,
process and engine) is consistent with our pricer.

## 2. The SVI surface in QuantLib (case C)

We fill a grid of strikes (4500 to 9500, step 100) and expiries (between 0.4 and 2.2 years) with the
SVI volatilities of each expiry, and pass it to `BlackVarianceSurface`. Everything else stays the
same: only the volatility input changes.

In [4]:
surface = pd.read_csv(os.path.join(DATA_DIR, SNAPSHOT + "_surface.csv"))

use = surface[(surface["T"] > 0.4) & (surface["T"] < 2.2)].reset_index(drop=True)   # expiries around 1 year

dates = []
for t in use["T"]:
    dates.append(today + int(round(t * 365)))       # from T in years to a date
strikes = list(np.arange(4500, 9501, 100.0))        # strike grid

# SVI matrix: rows = strikes, columns = dates
vols = Matrix(len(strikes), len(dates))
for j, row in use.iterrows():
    k = np.log(np.array(strikes) / row["F"])
    v = np.sqrt(svi(k, row["a"], row["b"], row["rho"], row["m"], row["sig"]) / row["T"])   # vector of 51 vols
    for i in range(len(strikes)):      # QuantLib needs a matrix, because a surface has two dimensions
        vols[i][j] = float(v[i])

surf_C = BlackVarianceSurface(today, NullCalendar(), dates, strikes, vols, dc)
surf_C.enableExtrapolation()
volTS_C = BlackVolTermStructureHandle(surf_C)

process_C = BlackScholesMertonProcess(QuoteHandle(u), dividendTS, riskFreeTS, volTS_C)
engine_C = AnalyticEuropeanEngine(process_C)

option.setPricingEngine(engine_C)
print("vol used:", volTS_C.blackVol(Date(26, 10, 2027), K))
print("price:", option.NPV())

vol used: 0.2133100923509162
price: 222.94843545548724


**Result.** With the SVI surface, QuantLib gives a volatility of 21.331% and a price of 222.95 for
the 90% put, against 21.329% and 222.90 in notebook 05. The small difference comes from the
interpolation between expiries, which QuantLib performs at fixed strike and on whole-day dates,
while our pricer works at fixed log-moneyness. It is about 0.002 volatility points, far below the
error of the surface.

With a constant volatility the same put costs 125.65, about 44% less than with the SVI surface. The
constant volatility ignores the skew: it prices a put at 90% of the spot with the at-the-money
volatility (16.70%) instead of the market volatility at that strike (21.33%).

## Next steps

- Case B: the same grid filled with market volatilities interpolated linearly between quotes, to
  measure the value of a smooth, arbitrage-free surface compared with a simple interpolation.
- Prices of hedging strategies (put, put spread, collar) with the three volatility inputs.